In [0]:
dbutils.widgets.removeAll()

In [0]:
!pip install bertopic

In [0]:
dbutils.library.restartPython()

In [0]:
%run ./helper

In [0]:
dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)
dbutils.widgets.text("input_table_name", input_table_list[0])

catalog=dbutils.widgets.get("source_catalog")
schema=dbutils.widgets.get("source_schema")
input_table_name=dbutils.widgets.get("input_table_name")

df = spark.read.table(f"{catalog}.{schema}.{input_table_name}")
table_name = f"{catalog}.{schema}.{input_table_name}"
print(f"table_name:{table_name}")


In [0]:
df

In [0]:
pddf=df.toPandas()

In [0]:
import numpy as np
embeddings = np.array(pddf["text_embedding"].tolist())
docs=pddf["text"].tolist()

In [0]:

from hdbscan import HDBSCAN
from umap import UMAP
from bertopic import BERTopic

# 1. Dimensionality Reduction (Crucial for HDBSCAN)
# HDBSCAN struggles with 768-dimension vectors, so we use UMAP to shrink them down to 5 dimensions first.
print("Configuring UMAP...")
umap_model = UMAP(n_neighbors=15, n_components=7, min_dist=0.0, metric='cosine', random_state=42)

# 2. Configure HDBSCAN
# We set min_cluster_size=100 to force the model to only create broad, massive topics.
print("Configuring HDBSCAN...")
hdbscan_model = HDBSCAN(
    min_cluster_size=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# 3. Initialize BERTopic with your custom models
print("Initializing BERTopic...")
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="arabic"
)

# 4. Fit the model using your PRE-CALCULATED MARBERTv2 embeddings
# This skips the heavy transformer step and runs in seconds!
print("Fitting the model...")
topics, probabilities = topic_model.fit_transform(docs, embeddings=embeddings)

# 5. View your new, generalized topics
print("Top Topics Discovered:")
print(topic_model.get_topic_info().head(10))